# NB03 · Session cells (run one at a time)
These cards end, exit or restart the Spark session, so they can't sit inside *Run all*. Run each cell on its own and check the result described above it.

### 1 · Native engine for one notebook (page 24)
`%%configure` must be the **first cell** of a session. Run this cell first in a fresh session, then check: `spark.conf.get("spark.native.enabled")` should return `true`.

In [ ]:
%%configure
{
    "conf": {
        "spark.native.enabled": "true"
    }
}

In [ ]:
spark.conf.get("spark.native.enabled")

### 2 · Right-size the pool (page 42)
Same rule: first cell of a fresh session (or stop the session, then run this). Check in the Spark UI that no more than 4 executors start.

In [ ]:
%%configure
{
  "conf": {
    "spark.dynamicAllocation.maxExecutors": "4"
  }
}

### 3 · Load only what changed (page 42)
Runs normally; if there are no new rows the notebook exits on purpose with "no new rows".

In [ ]:
%run NB00_Setup_Sample_Data

In [ ]:
# TEST SETUP (not in the book): inputs for the next card

def upsert_orders(src):
    print("would MERGE", src.count(), "rows")

In [ ]:
last = spark.sql(
    "SELECT MAX(updated_at) AS m "
    "FROM silver_orders"
).first().m

src = spark.read.table("bronze_orders")
if last is not None:
    src = src.filter(
        F.col("updated_at") > F.lit(last))

if src.isEmpty():
    notebookutils.notebook.exit("no new rows")

upsert_orders(src)   # MERGE, page 12

### 4 · Return a value from a notebook (page 10)
In an interactive session `exit` stops the run here but keeps the session open.

In [ ]:
nb.exit("done")               # return a string

### 5 · Stop the session (pages 10, 33, 42)
Run last. The session ends and stops using capacity; the status bar shows it stopped.

In [ ]:
# last cell of interactive work
notebookutils.session.stop()